> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Scalar subquery in WHERE

In [ ]:
-- Compare each amount to the overall average across all statuses.
SELECT transaction_id,
customer_id,
amount,
status
FROM payment_transactions
WHERE amount >
(
    SELECT AVG(amount)
    FROM payment_transactions
) 
ORDER BY amount DESC;

## Subquery using IN

In [ ]:
-- IN keeps payments whose customer is registered in Nigeria.
SELECT
    transaction_id,
    customer_id,
    amount,
    status
FROM payment_transactions
WHERE
    customer_id IN (
        SELECT customer_id
        FROM customers
        WHERE
            country = 'Nigeria'
    )
ORDER BY transaction_id;

## Subquery using NOT IN

In [ ]:
-- Exclude NULLs inside NOT IN to avoid unknown comparisons; NULL outer IDs would still be excluded.
SELECT transaction_id, customer_id, amount, status
FROM payment_transactions
WHERE customer_id NOT IN(
    SELECT customer_id
    FROM customers
    WHERE customer_id IS NOT NULL
)
ORDER BY transaction_id;

## Subquery inside HAVING

### Business problem: Identify customers whose total successful payment amount exceeds the average individual successful transaction amount

In [ ]:
-- Compare each customer successful total to the average individual successful payment.
SELECT
    customer_id,
    COUNT(*) AS total_transactions,
    SUM(amount) AS total_amount
FROM payment_transactions
WHERE
    status = 'Success'
GROUP BY
    customer_id
HAVING
    SUM(amount) > (
        SELECT AVG(amount)
        FROM payment_transactions
        WHERE
            status = 'Success'
    )
ORDER BY total_amount DESC;

## Subquery inside SELECT

### Business problem: Display every transaction alongside the overall average transaction amount and its deviation from that average.

In [ ]:
-- Scalar subqueries return the same overall average beside each payment and its signed difference.
SELECT
    transaction_id,
    amount,
    (
        SELECT AVG(amount)
        FROM payment_transactions
    ) AS overall_average,
    amount - (
        SELECT AVG(amount)
        FROM payment_transactions
    ) AS amount_difference
FROM payment_transactions
ORDER BY transaction_id;